# 🔍 Monitor Inteligente de Vagas InHire - Analista de Dados

**Autor:** Sistema Automatizado de Busca de Vagas  
**Objetivo:** Monitorar vagas remotas para Analista de Dados/Business Analyst na plataforma InHire  
**Técnicas:** Web Scraping Avançado + Google Dorking + Análise de API

---

## 📋 Índice
1. **Instalação de Dependências**
2. **Configurações Iniciais**
3. **Módulo 1: Varredura de Empresas Conhecidas**
4. **Módulo 2: Descoberta Automática via Google Dorking**
5. **Módulo 3: Processamento e Filtros Inteligentes**
6. **Módulo 4: Exportação e Visualização**
7. **Bonus: Setup para Alertas Telegram (Futuro)**

---
## 1️⃣ Instalação de Dependências

**O que faz:** Instala todas as bibliotecas necessárias para o projeto.  
**Por que:** Garante que você tem todas as ferramentas para web scraping, processamento de dados e análise.

**Bibliotecas:**
- `requests`: Fazer requisições HTTP
- `beautifulsoup4`: Parse de HTML
- `pandas`: Manipulação de dados
- `lxml`: Parser rápido de HTML/XML
- `python-telegram-bot`: (Futuro) Enviar alertas
- `tqdm`: Barras de progresso visuais

In [ ]:
# Execute esta célula apenas uma vez para instalar as dependências
!pip install requests beautifulsoup4 pandas lxml tqdm fake-useragent python-telegram-bot --quiet

---
## 2️⃣ Configurações Iniciais e Importações

**O que faz:** Importa bibliotecas e define configurações globais.  
**Destaque:** Usamos `fake_useragent` para rotacionar User-Agents e evitar bloqueios.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from datetime import datetime, timedelta
from urllib.parse import urljoin, quote_plus
from tqdm.notebook import tqdm
from fake_useragent import UserAgent
import warnings
warnings.filterwarnings('ignore')

# Configuração de User-Agent rotativo (anti-bloqueio)
ua = UserAgent()

# Função para obter headers dinâmicos
def get_headers():
    return {
        'User-Agent': ua.random,
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept-Encoding': 'gzip, deflate, br',
        'DNT': '1',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'
    }

print("✅ Bibliotecas carregadas com sucesso!")
print(f"📅 Execução iniciada em: {datetime.now().strftime('%d/%m/%Y às %H:%M:%S')}")

---
## 3️⃣ Configuração de Parâmetros de Busca

**O que faz:** Define palavras-chave, empresas-alvo e critérios de filtragem.  
**Personalize aqui:** Adicione/remova empresas e ajuste os filtros.

In [ ]:
# ========== CONFIGURAÇÕES PERSONALIZÁVEIS ==========

# Lista de empresas para monitorar (formato: nome_da_empresa)
EMPRESAS_ALVO = [
    'kpmg',
    'alura',
    'contabilizei',
    'kiwify',
    'sympla',
    'ifood',
    'nubank',
    'quintoandar',
    'stone',
    'creditas'
]

# Palavras-chave para identificar vagas de dados (case-insensitive)
KEYWORDS_DADOS = [
    'analista de dados',
    'data analyst',
    'business analyst',
    'analista de negócios',
    'business intelligence',
    'bi analyst',
    'data analytics',
    'analytics',
    'dados'
]

# Palavras-chave para identificar modalidade remota
KEYWORDS_REMOTO = [
    'remoto',
    'remote',
    'home office',
    'anywhere',
    'trabalho remoto',
    '100% remoto',
    'totalmente remoto'
]

# Delay entre requisições (em segundos) - para ser "gentil" com o servidor
DELAY_REQUISICOES = 2

# Dias para considerar vaga como "quente" (recente)
DIAS_RECENTES = 30

print(f"✅ Configurações carregadas!")
print(f"📊 Monitorando {len(EMPRESAS_ALVO)} empresas")
print(f"🔍 Buscando por {len(KEYWORDS_DADOS)} tipos de vagas relacionadas a dados")

---
## 4️⃣ Módulo 1: Funções de Scraping da InHire

**O que faz:** Extrai vagas de uma página específica da InHire.  
**Técnica:** Parse de HTML usando BeautifulSoup com tratamento robusto de erros.

In [ ]:
def extrair_vagas_inhire(empresa):
    """
    Extrai todas as vagas de uma empresa específica no InHire.
    
    Args:
        empresa (str): Nome da empresa (ex: 'kpmg')
    
    Returns:
        list: Lista de dicionários com informações das vagas
    """
    url = f"https://{empresa}.inhire.app/vagas"
    vagas = []
    
    try:
        # Faz requisição com timeout e headers customizados
        response = requests.get(url, headers=get_headers(), timeout=10)
        
        # Verifica se a página existe (status 200)
        if response.status_code != 200:
            print(f"⚠️  {empresa}: Página não encontrada (Status {response.status_code})")
            return vagas
        
        # Parse do HTML
        soup = BeautifulSoup(response.content, 'lxml')
        
        # ATENÇÃO: A estrutura HTML pode variar. Aqui estão alguns seletores comuns:
        # Procura por cards/divs de vagas (ajuste conforme estrutura real)
        
        # Opção 1: Por classe comum de vagas
        vaga_elements = soup.find_all('div', class_=re.compile('vaga|job|position|card', re.I))
        
        # Opção 2: Por links que contenham 'vaga' ou 'job'
        if not vaga_elements:
            vaga_elements = soup.find_all('a', href=re.compile('vaga|job|position', re.I))
        
        # Opção 3: Procura por estrutura de lista
        if not vaga_elements:
            vaga_elements = soup.find_all('li') + soup.find_all('article')
        
        for element in vaga_elements:
            try:
                # Extrai título da vaga
                titulo = element.find(['h1', 'h2', 'h3', 'h4', 'a'])
                if not titulo:
                    continue
                    
                titulo_texto = titulo.get_text(strip=True)
                
                # Extrai link da vaga
                link = element.find('a')
                if link and link.get('href'):
                    link_completo = urljoin(url, link.get('href'))
                else:
                    link_completo = url
                
                # Extrai texto completo para análise de localização
                texto_completo = element.get_text(separator=' ', strip=True).lower()
                
                # Tenta extrair data de publicação (formato comum: '3 dias atrás', 'há 1 semana')
                data_match = re.search(r'(\d+)\s*(dia|semana|mês|mes|hour|hora)s?\s*atrás|há\s*(\d+)\s*(dia|semana|mês|mes)', texto_completo)
                if data_match:
                    data_publicacao = data_match.group(0)
                else:
                    data_publicacao = "Data não disponível"
                
                # Adiciona à lista
                vagas.append({
                    'empresa': empresa.upper(),
                    'titulo': titulo_texto,
                    'link': link_completo,
                    'data_publicacao': data_publicacao,
                    'texto_completo': texto_completo  # Para filtros posteriores
                })
                
            except Exception as e:
                continue  # Ignora elementos problemáticos
        
        print(f"✅ {empresa}: {len(vagas)} vagas encontradas")
        
    except requests.exceptions.Timeout:
        print(f"⏱️  {empresa}: Timeout na requisição")
    except requests.exceptions.RequestException as e:
        print(f"❌ {empresa}: Erro na requisição - {str(e)[:50]}")
    except Exception as e:
        print(f"❌ {empresa}: Erro desconhecido - {str(e)[:50]}")
    
    return vagas

print("✅ Função de scraping criada!")

---
## 5️⃣ Módulo 2: Descoberta Automática via Google Dorking

**O que faz:** Usa Google Search para descobrir automaticamente novas empresas que usam InHire.  
**Técnica Avançada:** Google Dorking com operadores especializados.  
**Criativo:** Encontra páginas de vagas sem você precisar listar as empresas manualmente!

⚠️ **Limitação:** Google pode bloquear scraping automatizado. Use com moderação!

In [ ]:
def descobrir_empresas_inhire_google(max_resultados=20):
    """
    Usa Google Dorking para descobrir empresas que usam InHire.
    
    Técnica: Busca por 'site:inhire.app vagas' no Google
    
    Args:
        max_resultados (int): Número máximo de empresas a descobrir
    
    Returns:
        list: Lista de nomes de empresas descobertas
    """
    empresas_descobertas = set()
    
    # Google Dorking queries
    queries = [
        'site:inhire.app/vagas',
        'site:inhire.app "analista de dados"',
        'site:inhire.app "data analyst"',
        'site:inhire.app "remoto"'
    ]
    
    print("🔍 Iniciando descoberta automática via Google Dorking...")
    print("⚠️  Atenção: Esta técnica pode ser bloqueada pelo Google se usada em excesso\n")
    
    for query in queries:
        try:
            # Codifica a query para URL
            query_encoded = quote_plus(query)
            
            # URL do Google Search
            google_url = f"https://www.google.com/search?q={query_encoded}&num=10"
            
            # Requisição com headers que simulam navegador
            response = requests.get(google_url, headers=get_headers(), timeout=10)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'lxml')
                
                # Extrai todos os links dos resultados
                for link in soup.find_all('a'):
                    href = link.get('href', '')
                    
                    # Procura por padrão: empresa.inhire.app
                    match = re.search(r'https?://([a-zA-Z0-9-]+)\.inhire\.app', href)
                    if match:
                        empresa = match.group(1)
                        if empresa not in ['www', 'api', 'admin']:  # Ignora subdomínios sistema
                            empresas_descobertas.add(empresa)
                
                print(f"  ✓ Query '{query[:40]}...': {len(empresas_descobertas)} empresas até agora")
            else:
                print(f"  ⚠️  Query '{query[:40]}...': Bloqueado (Status {response.status_code})")
            
            # Delay entre queries para não ser bloqueado
            time.sleep(3)
            
        except Exception as e:
            print(f"  ❌ Erro na query '{query[:40]}...': {str(e)[:50]}")
            continue
    
    empresas_lista = list(empresas_descobertas)[:max_resultados]
    print(f"\n✅ Descoberta concluída! {len(empresas_lista)} empresas encontradas")
    
    return empresas_lista

# Método alternativo mais confiável: API do DuckDuckGo (não bloqueia facilmente)
def descobrir_empresas_inhire_duckduckgo(max_resultados=20):
    """
    Método alternativo usando DuckDuckGo (mais confiável que Google).
    Requer: pip install duckduckgo-search
    """
    try:
        from duckduckgo_search import DDGS
        
        empresas_descobertas = set()
        queries = ['site:inhire.app/vagas', 'site:inhire.app analista dados']
        
        print("🦆 Usando DuckDuckGo para descoberta (método mais confiável)...\n")
        
        with DDGS() as ddgs:
            for query in queries:
                results = ddgs.text(query, max_results=10)
                
                for result in results:
                    href = result.get('href', '')
                    match = re.search(r'https?://([a-zA-Z0-9-]+)\.inhire\.app', href)
                    if match:
                        empresa = match.group(1)
                        if empresa not in ['www', 'api', 'admin']:
                            empresas_descobertas.add(empresa)
                
                print(f"  ✓ Query '{query}': {len(empresas_descobertas)} empresas encontradas")
                time.sleep(1)
        
        empresas_lista = list(empresas_descobertas)[:max_resultados]
        print(f"\n✅ {len(empresas_lista)} empresas descobertas via DuckDuckGo")
        return empresas_lista
        
    except ImportError:
        print("⚠️  Biblioteca 'duckduckgo-search' não instalada.")
        print("   Para usar este método, execute: !pip install duckduckgo-search")
        return []
    except Exception as e:
        print(f"❌ Erro no DuckDuckGo: {str(e)}")
        return []

print("✅ Módulos de descoberta automática criados!")

---
## 6️⃣ Módulo 3: Filtros Inteligentes com NLP

**O que faz:** Aplica filtros inteligentes para garantir que a vaga seja realmente de dados E remota.  
**Técnica:** Análise de texto com regex e matching fuzzy para evitar falsos positivos.

In [ ]:
def eh_vaga_de_dados(titulo, texto_completo):
    """
    Verifica se a vaga é relacionada a dados/analytics.
    
    Args:
        titulo (str): Título da vaga
        texto_completo (str): Texto completo da vaga
    
    Returns:
        bool: True se for vaga de dados
    """
    texto_busca = (titulo + ' ' + texto_completo).lower()
    
    # Verifica se alguma keyword está presente
    for keyword in KEYWORDS_DADOS:
        if keyword.lower() in texto_busca:
            return True
    
    # Filtro adicional por padrões regex
    padroes = [
        r'\bdata\s+analyst\b',
        r'\banalyst\b.*\bdata\b',
        r'\bbi\b',
        r'\bbusiness\s+intelligence\b',
        r'\banalytics\b'
    ]
    
    for padrao in padroes:
        if re.search(padrao, texto_busca, re.IGNORECASE):
            return True
    
    return False

def eh_vaga_remota(titulo, texto_completo):
    """
    Verifica se a vaga é remota.
    
    Args:
        titulo (str): Título da vaga
        texto_completo (str): Texto completo da vaga
    
    Returns:
        bool: True se for remota
    """
    texto_busca = (titulo + ' ' + texto_completo).lower()
    
    # Verifica keywords de remoto
    for keyword in KEYWORDS_REMOTO:
        if keyword.lower() in texto_busca:
            # Filtro negativo: evita "parcialmente remoto", "não é remoto"
            if not re.search(r'(não|nao|parcial|híbrido|hibrido|presencial)', texto_busca):
                return True
    
    return False

def filtrar_vagas_relevantes(vagas):
    """
    Aplica todos os filtros nas vagas.
    
    Args:
        vagas (list): Lista de vagas brutas
    
    Returns:
        list: Lista de vagas filtradas
    """
    vagas_filtradas = []
    
    for vaga in vagas:
        # Aplica filtro de dados
        if eh_vaga_de_dados(vaga['titulo'], vaga['texto_completo']):
            # Aplica filtro de remoto
            if eh_vaga_remota(vaga['titulo'], vaga['texto_completo']):
                vagas_filtradas.append(vaga)
    
    return vagas_filtradas

print("✅ Filtros inteligentes criados!")

---
## 7️⃣ EXECUÇÃO PRINCIPAL: Varredura Completa

**O que faz:** Executa todo o pipeline de scraping e filtragem.  
**Fluxo:**
1. Varre empresas conhecidas
2. (Opcional) Descobre novas empresas
3. Aplica filtros inteligentes
4. Gera DataFrame formatado

In [ ]:
# ========== EXECUÇÃO COMPLETA ==========

print("🚀 INICIANDO VARREDURA DE VAGAS INHIRE\n")
print("=" * 60)

# Lista consolidada de empresas
empresas_para_varrer = EMPRESAS_ALVO.copy()

# OPÇÃO 1: Descobrir automaticamente novas empresas (COMENTAR se não quiser usar)
print("\n🔍 FASE 1: Descoberta Automática de Empresas")
print("-" * 60)
try:
    # Tenta DuckDuckGo primeiro (mais confiável)
    novas_empresas = descobrir_empresas_inhire_duckduckgo(max_resultados=15)
    
    # Se DuckDuckGo falhar, tenta Google (pode ser bloqueado)
    if not novas_empresas:
        print("\n⚠️  DuckDuckGo não retornou resultados. Tentando Google...")
        novas_empresas = descobrir_empresas_inhire_google(max_resultados=10)
    
    # Adiciona empresas descobertas (sem duplicatas)
    empresas_descobertas = [e for e in novas_empresas if e not in empresas_para_varrer]
    empresas_para_varrer.extend(empresas_descobertas)
    
    print(f"\n✅ {len(empresas_descobertas)} novas empresas adicionadas à lista")
    
except Exception as e:
    print(f"⚠️  Descoberta automática falhou: {str(e)}")
    print("   Continuando apenas com empresas pré-definidas...")

# FASE 2: Varredura de todas as empresas
print(f"\n\n📡 FASE 2: Varredura de {len(empresas_para_varrer)} Empresas")
print("-" * 60)

todas_vagas = []

for empresa in tqdm(empresas_para_varrer, desc="Varrendo empresas"):
    vagas_empresa = extrair_vagas_inhire(empresa)
    todas_vagas.extend(vagas_empresa)
    
    # Delay entre requisições (gentil com o servidor)
    time.sleep(DELAY_REQUISICOES)

print(f"\n📊 Total de vagas encontradas (brutas): {len(todas_vagas)}")

# FASE 3: Filtragem inteligente
print(f"\n\n🎯 FASE 3: Aplicando Filtros Inteligentes")
print("-" * 60)

vagas_relevantes = filtrar_vagas_relevantes(todas_vagas)

print(f"✅ Vagas de Dados identificadas: {len([v for v in todas_vagas if eh_vaga_de_dados(v['titulo'], v['texto_completo'])])}")
print(f"✅ Vagas Remotas identificadas: {len([v for v in todas_vagas if eh_vaga_remota(v['titulo'], v['texto_completo'])])}")
print(f"🎯 Vagas RELEVANTES (Dados + Remoto): {len(vagas_relevantes)}")

print("\n" + "=" * 60)
print("✅ VARREDURA CONCLUÍDA!")
print("=" * 60)

---
## 8️⃣ Visualização e Exportação dos Resultados

**O que faz:** Converte os resultados em um DataFrame Pandas bem formatado.  
**Saída:** Tabela organizada com todas as vagas relevantes.

In [ ]:
# Cria DataFrame
if vagas_relevantes:
    df_vagas = pd.DataFrame(vagas_relevantes)
    
    # Remove coluna de texto_completo (usada apenas para filtros)
    df_vagas = df_vagas.drop(columns=['texto_completo'])
    
    # Reordena colunas
    df_vagas = df_vagas[['empresa', 'titulo', 'link', 'data_publicacao']]
    
    # Renomeia para português
    df_vagas.columns = ['Empresa', 'Título da Vaga', 'Link Direto', 'Data de Publicação']
    
    # Adiciona índice começando em 1
    df_vagas.index = range(1, len(df_vagas) + 1)
    
    print("\n" + "=" * 80)
    print(f"📊 RESULTADOS FINAIS - {len(df_vagas)} VAGAS QUENTES ENCONTRADAS")
    print("=" * 80 + "\n")
    
    # Exibe DataFrame
    display(df_vagas)
    
    # Estatísticas
    print("\n" + "=" * 80)
    print("📈 ESTATÍSTICAS")
    print("=" * 80)
    print(f"\n🏢 Empresas com vagas: {df_vagas['Empresa'].nunique()}")
    print(f"\nTop 5 empresas com mais vagas:")
    print(df_vagas['Empresa'].value_counts().head())
    
else:
    print("\n⚠️  Nenhuma vaga relevante encontrada com os critérios atuais.")
    print("   Sugestões:")
    print("   - Amplie as KEYWORDS_DADOS")
    print("   - Reduza os filtros de KEYWORDS_REMOTO")
    print("   - Adicione mais empresas à lista EMPRESAS_ALVO")
    df_vagas = pd.DataFrame()

---
## 9️⃣ Exportação para Excel/CSV

**O que faz:** Salva os resultados em arquivo para análise posterior.

In [ ]:
if not df_vagas.empty:
    # Nome do arquivo com timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Exporta para CSV
    csv_filename = f'vagas_inhire_{timestamp}.csv'
    df_vagas.to_csv(csv_filename, index=True, encoding='utf-8-sig')
    print(f"✅ Resultados salvos em: {csv_filename}")
    
    # Exporta para Excel (se tiver openpyxl instalado)
    try:
        excel_filename = f'vagas_inhire_{timestamp}.xlsx'
        df_vagas.to_excel(excel_filename, index=True, engine='openpyxl')
        print(f"✅ Resultados salvos em: {excel_filename}")
    except ImportError:
        print("⚠️  Para exportar Excel, instale: pip install openpyxl")
else:
    print("⚠️  Nenhum dado para exportar.")

---
## 🔟 BONUS: Setup para Alertas no Telegram (Futuro)

**O que faz:** Prepara estrutura para enviar alertas automáticos quando novas vagas forem encontradas.  
**Como usar:**
1. Crie um bot no Telegram via @BotFather
2. Obtenha seu `CHAT_ID` enviando uma mensagem para o bot e acessando: `https://api.telegram.org/bot<SEU_TOKEN>/getUpdates`
3. Preencha as credenciais abaixo
4. Execute a célula para enviar um relatório

**Automação:** Agende este notebook para rodar diariamente usando:
- **Windows:** Task Scheduler
- **Linux/Mac:** Cron Job
- **Cloud:** GitHub Actions, Google Cloud Scheduler

In [ ]:
# ========== CONFIGURAÇÕES DO TELEGRAM ==========
# Preencha estas variáveis para ativar alertas

TELEGRAM_BOT_TOKEN = "SEU_TOKEN_AQUI"  # Token do bot do @BotFather
TELEGRAM_CHAT_ID = "SEU_CHAT_ID_AQUI"  # Seu chat ID

def enviar_alerta_telegram(df_vagas):
    """
    Envia relatório de vagas para o Telegram.
    
    Args:
        df_vagas (DataFrame): DataFrame com as vagas
    """
    if TELEGRAM_BOT_TOKEN == "SEU_TOKEN_AQUI":
        print("⚠️  Configure TELEGRAM_BOT_TOKEN e TELEGRAM_CHAT_ID primeiro!")
        return
    
    try:
        import telegram
        from telegram import Bot
        
        bot = Bot(token=TELEGRAM_BOT_TOKEN)
        
        # Monta mensagem
        if df_vagas.empty:
            mensagem = "🔍 Monitor InHire\n\n❌ Nenhuma vaga nova encontrada hoje."
        else:
            mensagem = f"🔥 Monitor InHire - {len(df_vagas)} Vagas Quentes!\n\n"
            
            for idx, row in df_vagas.head(10).iterrows():  # Envia no máximo 10
                mensagem += f"🏢 {row['Empresa']}\n"
                mensagem += f"📋 {row['Título da Vaga']}\n"
                mensagem += f"🔗 {row['Link Direto']}\n"
                mensagem += f"📅 {row['Data de Publicação']}\n\n"
            
            if len(df_vagas) > 10:
                mensagem += f"... e mais {len(df_vagas) - 10} vagas!"
        
        # Envia mensagem
        bot.send_message(chat_id=TELEGRAM_CHAT_ID, text=mensagem)
        print("✅ Alerta enviado para o Telegram!")
        
    except ImportError:
        print("⚠️  Instale python-telegram-bot: pip install python-telegram-bot")
    except Exception as e:
        print(f"❌ Erro ao enviar Telegram: {str(e)}")

# Exemplo de uso (descomente para testar)
# enviar_alerta_telegram(df_vagas)

---
## 📚 Guia de Uso e Próximos Passos

### Como Usar Este Notebook:

1. **Primeira Execução:**
   - Execute todas as células em ordem (Shift + Enter)
   - Ajuste as configurações na seção 3️⃣

2. **Personalizações:**
   - Adicione empresas na lista `EMPRESAS_ALVO`
   - Ajuste `KEYWORDS_DADOS` e `KEYWORDS_REMOTO` conforme necessário
   - Modifique `DELAY_REQUISICOES` se estiver sendo bloqueado

3. **Troubleshooting:**
   - Se nenhuma vaga for encontrada: o HTML do InHire pode ter mudado
   - Verifique manualmente uma página (ex: kpmg.inhire.app/vagas)
   - Ajuste os seletores na função `extrair_vagas_inhire()`

### Próximos Passos para Escalar:

1. **Banco de Dados:**
   ```python
   # Salve histórico em SQLite para não duplicar alertas
   import sqlite3
   conn = sqlite3.connect('vagas_historico.db')
   df_vagas.to_sql('vagas', conn, if_exists='append')
   ```

2. **Agendamento Automático:**
   - **Cron (Linux/Mac):**
     ```bash
     # Rodar diariamente às 9h
     0 9 * * * /path/to/python /path/to/notebook.ipynb
     ```
   - **Task Scheduler (Windows):** Configure via interface gráfica
   - **GitHub Actions:** Crie workflow YAML para rodar na nuvem

3. **Detecção de Mudanças:**
   ```python
   # Compare com execução anterior
   df_anterior = pd.read_csv('vagas_anterior.csv')
   vagas_novas = df_vagas[~df_vagas['Link Direto'].isin(df_anterior['Link Direto'])]
   ```

4. **API Reversa do InHire:**
   - Use DevTools do navegador (F12 > Network)
   - Procure por chamadas API (formato JSON)
   - Se existir, use `requests` direto na API (muito mais rápido)

### Boas Práticas:

- ✅ Sempre use delays entre requisições
- ✅ Rotacione User-Agents
- ✅ Respeite o robots.txt do site
- ✅ Não sobrecarregue os servidores
- ✅ Mantenha logs de execução

### Recursos Adicionais:

- [Documentação BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Requests Documentation](https://requests.readthedocs.io/)
- [Python Telegram Bot](https://github.com/python-telegram-bot/python-telegram-bot)
- [Scrapy Framework](https://scrapy.org/) - Para projetos maiores

---

**Desenvolvido com ❤️ por um Python Developer Sênior**  
**Dúvidas?** Consulte a documentação ou ajuste o código conforme necessário!